In [1]:
import heapq

class PuzzleState:
    def __init__(self, state, parent, move, depth, cost):
        self.state = state
        self.parent = parent
        self.move = move
        self.depth = depth
        self.cost = cost
        
    def __lt__(self, other):
        return self.cost < other.cost

class NPuzzleAStar:
    def __init__(self, n):
        self.n = n
        self.goal = tuple(list(range(1, n*n)) + [0]) # Đích: (1,2,3,...,0)
        
    # Heuristic: Khoảng cách Manhattan (Tối ưu hơn cách đếm ô sai vị trí của code mẫu)
    def manhattan_distance(self, state):
        dist = 0
        for i, val in enumerate(state):
            if val != 0:
                target_x, target_y = (val - 1) // self.n, (val - 1) % self.n
                curr_x, curr_y = i // self.n, i % self.n
                dist += abs(target_x - curr_x) + abs(target_y - curr_y)
        return dist

    def get_neighbors(self, state):
        neighbors = []
        zero_idx = state.index(0)
        x, y = zero_idx // self.n, zero_idx % self.n
        
        # Các hướng di chuyển: Lên, Xuống, Trái, Phải
        moves = {'Up': (-1, 0), 'Down': (1, 0), 'Left': (0, -1), 'Right': (0, 1)}
        
        for move_name, (dx, dy) in moves.items():
            nx, ny = x + dx, y + dy
            if 0 <= nx < self.n and 0 <= ny < self.n:
                new_idx = nx * self.n + ny
                new_state = list(state)
                # Đổi chỗ ô trống và ô kề
                new_state[zero_idx], new_state[new_idx] = new_state[new_idx], new_state[zero_idx]
                neighbors.append((tuple(new_state), move_name))
        return neighbors

    def solve(self, initial_state):
        # Biến đổi mảng 2D đầu vào thành tuple 1 chiều
        flat_initial = tuple(val for row in initial_state for val in row)
        
        open_list = []
        closed_set = set()
        
        start_node = PuzzleState(flat_initial, None, "Start", 0, self.manhattan_distance(flat_initial))
        heapq.heappush(open_list, start_node)
        
        while open_list:
            current_node = heapq.heappop(open_list)
            
            if current_node.state == self.goal:
                return self.build_path(current_node)
                
            closed_set.add(current_node.state)
            
            for next_state, move_name in self.get_neighbors(current_node.state):
                if next_state in closed_set:
                    continue
                    
                g = current_node.depth + 1
                h = self.manhattan_distance(next_state)
                f = g + h
                
                child_node = PuzzleState(next_state, current_node, move_name, g, f)
                heapq.heappush(open_list, child_node)
                
        return None # Không tìm thấy đường

    def build_path(self, node):
        path = []
        while node:
            path.append((node.move, node.state))
            node = node.parent
        return path[::-1]

    def print_state(self, state):
        for i in range(self.n):
            row = state[i*self.n : (i+1)*self.n]
            print(" ".join(f"{str(x):>2}" for x in row))
        print("-" * 10)

# Kịch bản chạy thử (Áp dụng cho n=4)
if __name__ == "__main__":
    initial = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 0, 11],
        [13, 14, 15, 12]
    ]
    
    solver = NPuzzleAStar(n=4)
    path = solver.solve(initial)
    
    if path:
        print(f"Đã tìm thấy đường đi sau {len(path)-1} bước:")
        for step, state in path:
            print(f"Bước: {step}")
            solver.print_state(state)
    else:
        print("Không có lời giải!")

Đã tìm thấy đường đi sau 2 bước:
Bước: Start
 1  2  3  4
 5  6  7  8
 9 10  0 11
13 14 15 12
----------
Bước: Right
 1  2  3  4
 5  6  7  8
 9 10 11  0
13 14 15 12
----------
Bước: Down
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14 15  0
----------


In [3]:
import heapq

class GraphRoutingAStar:
    def __init__(self, adjacency_list, heuristics):
        self.adj_list = adjacency_list
        self.h = heuristics

    def a_star_search(self, start, target):
        # Hàng đợi ưu tiên lưu (f_score, g_score, current_node)
        open_list = [(self.h[start], 0, start)]
        
        # Dictionary lưu đường đi và chi phí ngắn nhất
        came_from = {}
        g_score = {node: float('inf') for node in self.adj_list}
        g_score[start] = 0
        
        closed_set = set()

        while open_list:
            _, current_g, current_node = heapq.heappop(open_list)

            if current_node in closed_set:
                continue

            if current_node == target:
                return self.reconstruct_path(came_from, current_node)

            closed_set.add(current_node)

            # Duyệt các đỉnh kề
            for neighbor, weight in self.adj_list[current_node]:
                tentative_g_score = current_g + weight

                # Nếu tìm được đường đi tốt hơn
                if tentative_g_score < g_score.get(neighbor, float('inf')):
                    came_from[neighbor] = current_node
                    g_score[neighbor] = tentative_g_score
                    f_score = tentative_g_score + self.h.get(neighbor, 0)
                    heapq.heappush(open_list, (f_score, tentative_g_score, neighbor))

        return "Không tồn tại đường đi!"

    def reconstruct_path(self, came_from, current):
        path = [current]
        while current in came_from:
            current = came_from[current]
            path.append(current)
        return path[::-1]

# Kịch bản chạy thử đồ thị
if __name__ == "__main__":
    graph_data = {
        'A': [('B', 1), ('C', 3), ('D', 7)],
        'B': [('C', 1), ('D', 5)],
        'C': [('D', 12)],
        'D': []
    }
    
    heuristic_data = {'A': 3, 'B': 2, 'C': 1, 'D': 0}
    
    router = GraphRoutingAStar(graph_data, heuristic_data)
    result = router.a_star_search('A', 'D')
    print("Đường đi ngắn nhất từ A đến D là:", " -> ".join(result))

Đường đi ngắn nhất từ A đến D là: A -> B -> D


In [4]:
import heapq

class TSP_AStar:
    def __init__(self, graph):
        self.graph = graph
        self.nodes = list(graph.keys())
        self.num_nodes = len(self.nodes)

    # Heuristic: Trọng số cạnh nhỏ nhất kết nối đỉnh hiện tại với các đỉnh chưa thăm
    def heuristic(self, current, unvisited):
        if not unvisited:
            return self.graph[current].get(self.nodes[0], 0) # Chi phí quay về gốc
        
        min_cost = float('inf')
        for node in unvisited:
            if node in self.graph[current] and self.graph[current][node] < min_cost:
                min_cost = self.graph[current][node]
        return min_cost if min_cost != float('inf') else 0

    def solve(self, start_node):
        # Trạng thái: (f_score, g_score, current_node, visited_nodes_tuple)
        initial_visited = (start_node,)
        open_list = [(0, 0, start_node, initial_visited)]
        
        # Dùng dictionary lưu chi phí để tránh duyệt lại trạng thái y hệt
        best_g_scores = {(start_node, initial_visited): 0}

        while open_list:
            _, current_g, current, visited = heapq.heappop(open_list)

            # Nếu đã thăm đủ đỉnh, quay về gốc
            if len(visited) == self.num_nodes:
                if start_node in self.graph[current]:
                    final_cost = current_g + self.graph[current][start_node]
                    return visited + (start_node,), final_cost
                continue

            unvisited = [n for n in self.nodes if n not in visited]

            for neighbor in unvisited:
                if neighbor in self.graph[current]:
                    move_cost = self.graph[current][neighbor]
                    new_g = current_g + move_cost
                    new_visited = visited + (neighbor,)
                    
                    state_key = (neighbor, new_visited)
                    
                    if new_g < best_g_scores.get(state_key, float('inf')):
                        best_g_scores[state_key] = new_g
                        h = self.heuristic(neighbor, [n for n in self.nodes if n not in new_visited])
                        f = new_g + h
                        heapq.heappush(open_list, (f, new_g, neighbor, new_visited))

        return None, float('inf')

# Kịch bản chạy thử bài toán Giao hàng
if __name__ == "__main__":
    # Đồ thị biểu diễn khoảng cách giữa các điểm giao hàng
    tsp_graph = {
        'A': {'B': 10, 'C': 15, 'D': 20},
        'B': {'A': 10, 'C': 35, 'D': 25},
        'C': {'A': 15, 'B': 35, 'D': 30},
        'D': {'A': 20, 'B': 25, 'C': 30}
    }
    
    tsp_solver = TSP_AStar(tsp_graph)
    best_route, min_cost = tsp_solver.solve('A')
    
    print(f"Lộ trình người giao hàng tối ưu: {' -> '.join(best_route)}")
    print(f"Tổng chi phí quãng đường: {min_cost}")

Lộ trình người giao hàng tối ưu: A -> B -> D -> C -> A
Tổng chi phí quãng đường: 80


In [5]:
import copy
import heapq

# Kích thước bài toán (8-puzzle có n=3) 
N = 3

# Định nghĩa 4 hướng di chuyển của ô trống: Lên, Trái, Xuống, Phải 
dx = [-1, 0, 1, 0]
dy = [0, -1, 0, 1]

class PuzzleNode:
    """Cấu trúc của một đỉnh (node) trên cây tìm kiếm"""
    def __init__(self, board, parent, blank_x, blank_y, g_cost, h_cost):
        self.board = board
        self.parent = parent
        self.blank_x = blank_x
        self.blank_y = blank_y
        
        # Các hệ số của thuật toán A*
        self.g = g_cost  # Chi phí thực tế g(n) từ gốc đến điểm hiện tại
        self.h = h_cost  # Chi phí heuristic h(n) ước lượng đến đích
        self.f = self.g + self.h # Tổng chi phí f(n) = g(n) + h(n)

    # Nạp chồng toán tử nhỏ hơn '<' để hàng đợi ưu tiên (Min-Heap) biết cách sắp xếp 
    def __lt__(self, other):
        return self.f < other.f

def calculate_heuristic(current_board, goal_board):
    """Hàm ước tính chi phí h(n): Đếm số lượng ô nằm sai vị trí """
    misplaced = 0
    for i in range(N):
        for j in range(N):
            # Bỏ qua ô trống (ô có giá trị 0) trong lúc đếm
            if current_board[i][j] != 0 and current_board[i][j] != goal_board[i][j]:
                misplaced += 1
    return misplaced

def is_valid_pos(x, y):
    """Kiểm tra xem vị trí x, y có nằm trong bàn cờ không """
    return 0 <= x < N and 0 <= y < N

def print_board(board):
    """Hàm in ma trận trạng thái ra màn hình """
    for row in board:
        for val in row:
            print(f"{val} ", end="")
        print()
    print()

def print_solution_path(node):
    """Truy vết ngược từ đích về gốc để in ra các bước đi """
    if node is None:
        return
    print_solution_path(node.parent)
    print_board(node.board)

def solve_8_puzzle(start_board, goal_board, start_blank_x, start_blank_y):
    """Hàm chính thực thi thuật toán A*"""
    open_list = [] # Mảng chứa các đỉnh chờ được duyệt (OPEN) [cite: 41, 42]
    
    # Tính h(n) cho đỉnh xuất phát và tạo node gốc
    h_initial = calculate_heuristic(start_board, goal_board)
    root = PuzzleNode(start_board, None, start_blank_x, start_blank_y, 0, h_initial)
    
    # Đưa đỉnh gốc vào hàng đợi ưu tiên (Bước 1) [cite: 44]
    heapq.heappush(open_list, root)

    while open_list:
        # (Bước 2): Lấy ra đỉnh có f(n) nhỏ nhất [cite: 45]
        current = heapq.heappop(open_list)

        # Nếu h(n) == 0 tức là không còn ô nào sai vị trí -> Đã đến đích (Bước 2) [cite: 46]
        if current.h == 0:
            print_solution_path(current)
            return

        # (Bước 3): Tìm các đỉnh kế cận và đưa vào open_list [cite: 47]
        for i in range(4):
            new_x = current.blank_x + dx[i]
            new_y = current.blank_y + dy[i]

            if is_valid_pos(new_x, new_y):
                # Copy ma trận để tạo ra trạng thái mới
                new_board = copy.deepcopy(current.board)
                
                # Tráo đổi vị trí của ô trống và ô bên cạnh
                new_board[current.blank_x][current.blank_y], new_board[new_x][new_y] = \
                    new_board[new_x][new_y], new_board[current.blank_x][current.blank_y]
                
                # Tính các hệ số g, h cho trạng thái mới
                new_g = current.g + 1
                new_h = calculate_heuristic(new_board, goal_board)
                
                # Khởi tạo Node con và nhét vào hàng đợi ưu tiên
                child_node = PuzzleNode(new_board, current, new_x, new_y, new_g, new_h)
                heapq.heappush(open_list, child_node)

if __name__ == "__main__":
    # Khai báo dữ liệu y chang như trong tài liệu mẫu để kiểm chứng 
    initial_state = [
        [1, 2, 3],
        [5, 6, 0],
        [7, 8, 4]
    ]
    
    goal_state = [
        [1, 2, 3],
        [5, 8, 6],
        [0, 7, 4]
    ]
    
    # Trong ma trận initial_state, ô trống '0' nằm ở dòng index 1, cột index 2 
    solve_8_puzzle(initial_state, goal_state, 1, 2)

1 2 3 
5 6 0 
7 8 4 

1 2 3 
5 0 6 
7 8 4 

1 2 3 
5 8 6 
7 0 4 

1 2 3 
5 8 6 
0 7 4 

